<a href="https://colab.research.google.com/github/yunseo449/ESAA/blob/main/ESAA_YB_WEEK12_2_2_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#신용카드 고객 세그먼트 분류 AI 경진대회(DACON, 2025.03~04, 평가지표 Macro F1)
금융 데이터를 활용해 고객 별 일정 기간의 데이터를 바탕으로 카드사 신용카드 고객 세그먼트(Segment) 분류하는 AI 알고리즘을 개발하는 것이 목표

##1. 데이터
- 데이터 구성: 2018년 7~12월, 6개월치 월별 데이터가 8개 카테고리로 나뉘어 있음
  - 회원정보, 신용정보, 승인매출정보, 청구입금정보, 잔액정보, 채널정보, 마케팅정보, 성과정보
- ID: 고객 고유 ID (한 고객당 6개월 → 6행)
- 기준년월: 데이터 기준 연월
- Segment: 고객 등급 A~E (Target 변수)
- 크기: train 240만 행, test 60만 행. 8개 파일을 합치면 약 850개 컬럼.

##2. 코드 흐름
1. 파일 정리: 월별 parquet 파일을 카테고리별로 합치고, 8개 카테고리를 ID와 기준년월 기준으로 병합
2. 불필요한 컬럼 제거: 값이 전부 같은 컬럼과 내용이 중복되는 컬럼을 제거. 컬럼 수가 858개에서 746개로 줄어듦.
3. BASE 모델: A·B 고객을 빼고 C·D·E만으로 CatBoost를 학습(10-Fold).
4. VIP 모델: A·B 고객의 특징을 이용해 후보 고객을 골라내고, 그 후보들만 따로 학습해서 A·B를 찾음.
5. 결과 합치기: BASE 예측 결과에서 VIP 모델이 A·B로 찾은 고객만 덮어씌움.(2-way optimization)

##3. 주요 코드 분석
1. 상수·중복 컬럼 제거
```
constant_cols = [col for col in df.columns if df[col].nunique() == 1]
df = df.drop(columns=constant_cols)
```
-> 모든 행의 값이 같은 컬럼은 예측에 정보를 주지 않으니 제거함. 850개가 넘는 컬럼을 줄이는 첫 단계.

2. Base 모델: A·B 고객 제외 + C 클래스 가중치
```
train = train[~train['ID'].isin(ab_ids)]
'class_weights': [2, 1, 1]   # C, D, E
```
-> 수가 아주 적은 A·B는 Base 모델에서 빼고, 서로 잘 헷갈리는 C와 D 중 C에 가중치 2를 줌.(이 조정으로 점수가 의미 있게 올랐음)

3. 고객 단위 예측 (6개월 확률 평균)
```
mean_probs = prob_df.groupby('ID').mean()
mean_probs['Segment'] = mean_probs.values.argmax(axis=1)
```
-> 예측은 행(월) 단위로 나오지만 제출은 고객 단위이므로 한 고객의 6개월치 확률을 평균 내고, 확률이 가장 높은 등급을 고름.

4. VIP 후보 고르기: A 고객끼리 값이 같은 컬럼 이용
```
fixed_columns_A = {col: train_A[col].iloc[0] for col in cols_to_check if train_A[col].nunique() == 1}
matching_ids = matching_ids.groupby('ID').filter(lambda x: len(x) == 6)
```
-> A 고객 전원이 같은 값을 갖는 컬럼(83개)을 찾음. 그 값과 6개월 내내 일치하는 고객만 A 후보로 남기고 B도 같은 방식(121개 컬럼)

5. VIP 모델: 극단적인 클래스 가중치
```
'class_weights': [20, 50, 2, 1, 1]   # A, B, C, D, E
```
-> 후보 안에서도 A·B는 소수라서 가중치를 크게 줌. 그 결과 test에서 A 40명, B 1명을 찾아냄.

6. 최종 결합
```
base_df.loc[base_df['ID'].isin(a_ids), 'Segment'] = 'A'
base_df.loc[base_df['ID'].isin(b_ids), 'Segment'] = 'B'
```
-> Base 결과는 그대로 두고, VIP 모델이 찾은 고객만 A나 B로 바꿈.



##4. 새롭게 알게 된 점 / 배운 점
- Macro F1은 등급마다 F1을 따로 구한 뒤 평균을 낸다.
    - Macro F1 = (F1_A + F1_B + F1_C + F1_D + F1_E) ÷ 5
    - 그래서 인원이 아주 적은 A·B를 몇 명만 맞혀도 점수가 크게 오른다. 평가지표의 구조를 이해하고 전략을 짠 점이 인상적이었다.
- 하나의 모델로 모든 클래스를 맞히려 하지 않고, 다수 클래스용 모델과 소수 클래스 전용 모델을 나눈 '2단계' 구조를 배웠다.
- 작성자는 SMOTE나 Voting, 앙상블도 시도했지만 차이가 없어서 결국 CatBoost 단일 모델을 택했다. 기법을 많이 쌓는 것보다 문제 구조에 맞는 단순한 해법이 더 효과적일 수 있다는 걸 느꼈다.
- 한계: Optuna 탐색 코드는 주석 처리되어 있고 찾아둔 파라미터를 그대로 넣었다. VIP 모델의 가중치(20, 50 등)도 근거 없이 수동으로 정해서, 그대로 재현하거나 다른 데이터에 일반화하기는 어려울 수 있다.